# Chromadb를 활용한 DB SCHEMA 작성

#### 환경 설정

In [1]:
# 라이브러리와 한국어 임베딩 모델
import numpy as np
import pandas as pd
import json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb

# 지난 단원에서 배운 한국어 임베딩 모델 — 문장 한 개를 768차원 벡터로 바꿉니다.
# (처음 부를 때 모델을 내려받느라 조금 걸릴 수 있어요.)
emb_model = SentenceTransformer('jhgan/ko-sroberta-multitask')
print('임베딩 모델 준비 완료 — 벡터 차원:', emb_model.get_embedding_dimension())

c:\Users\Playdata\Desktop\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9882.86it/s]


임베딩 모델 준비 완료 — 벡터 차원: 768


#### 데이터 가져오기

In [ ]:
with open('../data/RAG/maple_guides_documents_chunked.json', encoding='utf-8') as f :
    doc_json = json.load(f)


doc_texts = doc_json.get['page_content']

doc_emb = emb_model.encode(doc_texts, normalize_embeddings=True)

TypeError: list indices must be integers or slices, not str

#### Chromadb 연결

In [ ]:
# 클라이언트 — 메모리에 살아서 커널을 끄면 사라진다
client = chromadb.EphemeralClient()

# 컬렉션 — get_or_create 라 여러 번 실행해도 안전하다
maple_collect = client.get_or_create_collection(
    'game_guide',
    metadata={'hnsw:space': 'cosine'},   # 거리 기준
)

# 적재 — 네 인자는 같은 위치끼리 한 문서를 이룬다
maple_collect.add(
    ids=doc_json['id'].tolist(),
    embeddings=doc_emb,                        # 우리 벡터를 직접 넣는다
    documents=doc_json['page_content'].tolist(),
    metadatas=[
        {'name': n, 'region': r, 'type': t, 'entrance_fee': int(f)}
        for n, r, t, f in zip(
            doc_json['name'], doc_json['region'], doc_json['type'], doc_json['entrance_fee']
        )
    ],
)

print("컬렉션에 저장된 문서 수:", doc_json.count())


#### Chromadb Schema
```JSON
{
    "source": "guide",                  // 출처 (guide, patch_note, faq 등)
    "name": "보스 레이드: 자쿰 가이드",     // 문서 원본 제목
    "section_title": "보스/레이드",       // 카테고리 (필터링 핵심 키)
    "article_id": 101,                  // 원본 게시글 ID (Integer 또는 String)
    "board_id": 1,                      // 게시판 ID
    "url": "https://maplestory...",     // 출처 링크 (답변 시 참조 URL 제공용)
    "chunk_index": 0,                   // 문서 내 청크 순서
    "total_chunks": 3                   // 문서 전체 청크 수
}
```

#### 의미 기반 검색 (임베딩 코사인 유사도)

In [ ]:
# 질문도 문서와 같은 방식으로 임베딩한다
query = ""



query_emb = emb_model.encode([query], normalize_embeddings=True)

# 결과가 1×N 행렬이라 [0] 으로 한 줄을 꺼낸다
sims = cosine_similarity(query_emb, doc_emb)[0]

# 부호를 뒤집어 argsort 하면 내림차순
top3 = np.argsort(-sims)[:3]

print("[의미 기반 검색] Top-3:")
for i in top3:
    print(f"  유사도 {sims[i]:.3f}  |  {spots.loc[i, 'name']} ({spots.loc[i, 'type']})")

#### Top-K 검색

In [ ]:
query = "바다에서 시원하게 물놀이하기 좋은 곳"
query_emb = emb_model.encode([query], normalize_embeddings=True)

res = travel_col.query(query_embeddings=query_emb, n_results=3)

print("질문:", query)

# 질문을 여러 개 넣을 수 있는 구조라 [0] 으로 한 겹 벗긴다
for doc_id, dist, meta in zip(
    res['ids'][0], res['distances'][0], res['metadatas'][0]
):
    print(f"  거리 {dist:.3f}  |  {doc_id}  {meta['name']} ({meta['type']})")

#### Qdrant

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

# 1) 클라이언트와 컬렉션
qclient = QdrantClient(':memory:')
dim = doc_emb.shape[1]   # Qdrant 는 차원을 미리 알려 줘야 한다

# create_collection 은 이미 있으면 에러 — 지우고 다시 만든다
if qclient.collection_exists('travel_guide'):
    qclient.delete_collection('travel_guide')

qclient.create_collection(
    'travel_guide',
    vectors_config=VectorParams(size=dim, distance=Distance.COSINE),
)

# 2) 적재 — PointStruct 하나가 문서 하나 (id·vector·payload)
qclient.upsert('travel_guide', points=[
    PointStruct(
        id=i,
        vector=doc_emb[i],
        payload={'name': spots.loc[i, 'name'], 'type': spots.loc[i, 'type']},
    )
    for i in range(len(spots))
])

print("Qdrant 에 저장된 점 개수:", qclient.count("travel_guide").count)

# 3) 검색 — ChromaDB 와 같은 질문
query = "바다에서 시원하게 물놀이하기 좋은 곳"
query_emb = emb_model.encode([query], normalize_embeddings=True)

# 질문 벡터를 리스트로 감싸지 않고 하나만 넘긴다
hits = qclient.query_points('travel_guide', query=query_emb[0], limit=3).points

print("\n[Qdrant] 질문:", query)

# score 는 Chroma 의 distance 와 반대 — 클수록 가깝다
for h in hits:
    print(f"  점수 {h.score:.3f}  |  {h.payload['name']} ({h.payload['type']})")